In [1]:
from io import BytesIO
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from IPython.display import clear_output, display
from PIL import Image

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
MODEL_PATH = PROJECT_ROOT / "models" / "traffic_sign_cnn.keras"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Trained model was not found: {MODEL_PATH.resolve()}")

model = tf.keras.models.load_model(MODEL_PATH)
print(f"Loaded model: {MODEL_PATH.resolve()}")

Loaded model: D:\traffic-sign-recogonition\models\traffic_sign_cnn.keras


In [ ]:
class_names = {
    0: "Speed limit (20 km/h)",
    1: "Speed limit (30 km/h)",
    2: "Speed limit (50 km/h)",
    3: "Speed limit (60 km/h)",
    4: "Speed limit (70 km/h)",
    5: "Speed limit (80 km/h)",
    6: "End of speed limit (80 km/h)",
    7: "Speed limit (100 km/h)",
    8: "Speed limit (120 km/h)",
    9: "No passing",
    10: "No passing by vehicles over 3.5 tons",
    11: "Right-of-way at the next intersection",
    12: "Priority road",
    13: "Yield",
    14: "Stop",
    15: "No vehicles",
    16: "Vehicles over 3.5 tons prohibited",
    17: "No entry",
    18: "General caution",
    19: "Dangerous curve to the left",
    20: "Dangerous curve to the right",
    21: "Double curve",
    22: "Bumpy road",
    23: "Slippery road",
    24: "Road narrows on the right",
    25: "Road work",
    26: "Traffic signals",
    27: "Pedestrians",
    28: "Children crossing",
    29: "Bicycles crossing",
    30: "Beware of ice or snow",
    31: "Wild animals crossing",
    32: "End of all speed and passing limits",
    33: "Turn right ahead",
    34: "Turn left ahead",
    35: "Ahead only",
    36: "Go straight or right",
    37: "Go straight or left",
    38: "Keep right",
    39: "Keep left",
    40: "Roundabout mandatory",
    41: "End of no passing",
    42: "End of no passing by vehicles over 3.5 tons",
}

if set(class_names) != set(range(43)):
    raise ValueError("class_names must contain exactly class IDs 0-42.")

TRAIN_DIR = PROJECT_ROOT / "dataset" / "Train"
representative_images = []
for class_id in range(43):
    class_images = sorted((TRAIN_DIR / str(class_id)).glob("*.png"))
    if not class_images:
        raise FileNotFoundError(f"No representative training image found for class {class_id}.")
    representative_images.append(class_images[0])

figure, axes = plt.subplots(7, 7, figsize=(15, 15))
for class_id, (axis, image_path) in enumerate(zip(axes.ravel(), representative_images)):
    with Image.open(image_path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"{class_id}: {class_names[class_id]}", fontsize=8)
    axis.axis("off")

figure.suptitle("Representative Training Images and Verified Class Names")
figure.tight_layout()
plt.show()
print("Verified all 43 names against representative images in dataset/Train/<class_id>/.")
print("The mapping matches the standard GTSRB classes for this dataset.")

In [ ]:
upload_widget = widgets.FileUpload(
    accept=".png,.jpg,.jpeg",
    multiple=False,
    description="Choose Files",
)
prediction_output = widgets.Output()

def get_uploaded_file(upload_value):
    if isinstance(upload_value, dict):
        return next(iter(upload_value.values()))
    return upload_value[0]

def predict_uploaded_image(change):
    if not upload_widget.value:
        return

    uploaded_file = get_uploaded_file(upload_widget.value)
    image_bytes = uploaded_file["content"]

    with prediction_output:
        clear_output(wait=True)

        with Image.open(BytesIO(image_bytes)) as uploaded_image:
            original_image = uploaded_image.copy()
            processed_image = uploaded_image.convert("RGB").resize((32, 32), Image.Resampling.LANCZOS)

        image_array = np.asarray(processed_image, dtype=np.float32) / 255.0
        image_batch = np.expand_dims(image_array, axis=0)
        prediction_probabilities = model.predict(image_batch, verbose=0)
        predicted_class = int(np.argmax(prediction_probabilities[0]))
        confidence = float(np.max(prediction_probabilities[0]))

        figure, axis = plt.subplots(figsize=(4, 4))
        axis.imshow(original_image)
        axis.axis("off")
        axis.set_title("Uploaded image")
        plt.show()

        print(f"Predicted class: {predicted_class}")
        print(f"Traffic sign: {class_names[predicted_class]}")
        print(f"Confidence: {confidence:.2%}")

upload_widget.observe(predict_uploaded_image, names="value")
display(widgets.VBox([upload_widget, prediction_output]))

The notebook uses the existing model only. It does not retrain, modify, save over, or replace the model, and it does not request a file path or human-readable class name.